In [27]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# -----------------------------
# CONFIG
# -----------------------------
BATCH_SIZE = 16
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(device)

# -----------------------------
# LOAD DATA
# -----------------------------
X_lig = np.load("data/processed_features/X_lig.npy", mmap_mode='r')
X_prot = np.load("data/processed_features/X_prot.npy", mmap_mode='r')
y = np.load("data/processed_features/y.npy", mmap_mode='r')

print("Ligand shape:", X_lig.shape)
print("Protein shape:", X_prot.shape)

# -----------------------------
# SPLIT
# -----------------------------
Xl_train, Xl_test, Xp_train, Xp_test, y_train, y_test = train_test_split(
    X_lig, X_prot, y, test_size=0.2, random_state=42
)

mps
Ligand shape: (49889, 1024)
Protein shape: (49889, 1024)


In [30]:
Xl_train
X_prot

memmap([[ 0.01661635, -0.03018764,  0.00090213, ..., -0.04382138,
         -0.00518697, -0.01704268],
        [ 0.01661635, -0.03018764,  0.00090213, ..., -0.04382138,
         -0.00518697, -0.01704268],
        [ 0.04728722,  0.04910565,  0.01601403, ..., -0.04229094,
          0.01616114,  0.02428306],
        ...,
        [ 0.06391568,  0.00758111, -0.00317459, ..., -0.01832715,
         -0.01082724, -0.01912413],
        [ 0.15468745,  0.08146743,  0.02502209, ..., -0.07279287,
         -0.08447044, -0.00546397],
        [ 0.06391568,  0.00758111, -0.00317459, ..., -0.01832715,
         -0.01082724, -0.01912413]], shape=(49889, 1024), dtype=float32)

In [ ]:
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score


# -----------------------------
# CONFIG
# -----------------------------
FP_SIZE = 1024
MAX_LEN = 300
BATCH_SIZE = 16
EPOCHS = 10

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# -----------------------------
# DATASET
# -----------------------------
class DTIDataset(Dataset):
    def __init__(self, lig, prot, y):
        self.lig = lig
        self.prot = prot
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.lig[idx], dtype=torch.float32),
            torch.tensor(self.prot[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.float32),
        )


train_loader = DataLoader(DTIDataset(Xl_train, Xp_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True)

test_loader = DataLoader(DTIDataset(Xl_test, Xp_test, y_test),
                         batch_size=BATCH_SIZE)


# -----------------------------
# MODEL
# -----------------------------
class DTIModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.ligand_net = nn.Sequential(
            nn.Linear(FP_SIZE, 256),
            nn.ReLU(),
            nn.Linear(256, 64)
        )

        self.embedding = nn.Embedding(21, 64, padding_idx=0)

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, ligand, protein):
        l = self.ligand_net(ligand)

        p = self.embedding(protein)
        p = p.mean(dim=1)

        x = torch.cat([l, p], dim=1)
        return self.fc(x).squeeze()


model = DTIModel().to(device)


# -----------------------------
# LOSS (IMPORTANT)
# -----------------------------
pos = torch.tensor(np.sum(y_train == 1))
neg = torch.tensor(np.sum(y_train == 0))

pos_weight = torch.tensor([neg / pos]).to(device)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


# -----------------------------
# TRAIN
# -----------------------------
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for ligand, protein, label in train_loader:
        ligand = ligand.to(device)
        protein = protein.to(device)
        label = label.to(device)

        pred = model(ligand, protein)

        loss = loss_fn(pred, label)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


# -----------------------------
# EVALUATION
# -----------------------------
model.eval()
preds, probs, true = [], [], []

with torch.no_grad():
    for ligand, protein, label in test_loader:
        ligand = ligand.to(device)
        protein = protein.to(device)

        pred = model(ligand, protein)
        prob = torch.sigmoid(pred)

        preds.extend((prob > 0.5).cpu().numpy())
        probs.extend(prob.cpu().numpy())
        true.extend(label.numpy())

acc = accuracy_score(true, preds)
auc = roc_auc_score(true, probs)

print("Accuracy:", acc)
print("ROC-AUC:", auc)

Using device: mps
Epoch 1, Loss: 1689.4960
Epoch 2, Loss: 1460.8979
Epoch 3, Loss: 1291.1957
Epoch 4, Loss: 1164.1373
Epoch 5, Loss: 1071.3490
Epoch 6, Loss: 1003.5612
Epoch 7, Loss: 955.2441
Epoch 8, Loss: 907.7053
Epoch 9, Loss: 879.8085
Epoch 10, Loss: 858.2718
Accuracy: 0.7445379835638405
ROC-AUC: 0.830112729662605
